In [ ]:
# Import package
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import igraph
import scvelo as scv
import loompy as lmp
import anndata as ad
import harmonypy as hm
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import celltypist
from celltypist import models
from celltypist.models import Model

In [ ]:
adata=sc.read("/home/jaehyunchoi/Liver_bio_project/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_adata_mono02.h5ad")
adata

In [ ]:
adata.obs['monocluster'] = adata.obs['10_cluster'].replace({
    '0': 'mono0',
    '1': 'mono1',
    '2': 'mono2',
    '3': 'mono3',
    '4': 'mono4',
    '5': 'mono5',
    '6': 'mono6',
    '7': 'mono7',
    '8': 'mono8',
    '9': 'mono9'
})

sc.pl.umap(adata, color=['monocluster'])

## Glycolysis & Mitochondrial Energy Activity Analysis per monocluster
---

In [ ]:
# ─── Gene sets ───────────────────────────────────────────────────
glycolysis_genes = [
    'HK1', 'HK2', 'GPI', 'PFKL', 'PFKM', 'PFKP',
    'ALDOA', 'ALDOB', 'ALDOC', 'TPI1', 'GAPDH',
    'PGK1', 'PGAM1', 'ENO1', 'ENO2', 'PKM', 'LDHA', 'LDHB'
]

mito_energy_genes = [
    # Complex I (NADH dehydrogenase)
    'NDUFV1', 'NDUFV2', 'NDUFS1', 'NDUFS2', 'NDUFS3', 'NDUFS4',
    'NDUFS7', 'NDUFS8', 'NDUFB1', 'NDUFB2', 'NDUFB3', 'NDUFB4',
    # Complex II (Succinate dehydrogenase)
    'SDHA', 'SDHB', 'SDHC', 'SDHD',
    # Complex III (Cytochrome bc1)
    'UQCRC1', 'UQCRC2', 'UQCRB', 'CYC1',
    # Complex IV (Cytochrome c oxidase)
    'COX4I1', 'COX5A', 'COX5B', 'COX6A1', 'COX7A2',
    # Complex V (ATP synthase)
    'ATP5F1A', 'ATP5F1B', 'ATP5F1C', 'ATP5F1D', 'ATP5F1E'
]

# Filter to genes present in adata
glycolysis_genes = [g for g in glycolysis_genes if g in adata.var_names]
mito_energy_genes = [g for g in mito_energy_genes if g in adata.var_names]

print(f'Glycolysis genes found: {len(glycolysis_genes)} -> {glycolysis_genes}')
print(f'Mito energy genes found: {len(mito_energy_genes)} -> {mito_energy_genes}')

# Score gene sets
sc.tl.score_genes(adata, gene_list=glycolysis_genes, score_name='glycolysis_score')
sc.tl.score_genes(adata, gene_list=mito_energy_genes, score_name='mito_energy_score')

adata.obs[['monocluster', 'glycolysis_score', 'mito_energy_score']].groupby('monocluster').agg(['mean', 'std'])

In [ ]:
# UMAP: glycolysis & mito energy score 시각화
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sc.pl.umap(adata, color='monocluster', ax=axes[0], show=False, frameon=False, title='monocluster')
sc.pl.umap(adata, color='glycolysis_score', ax=axes[1], show=False, frameon=False,
           title='Glycolysis Score', color_map='RdYlBu_r')
sc.pl.umap(adata, color='mito_energy_score', ax=axes[2], show=False, frameon=False,
           title='Mito Energy (OXPHOS) Score', color_map='RdYlBu_r')

plt.tight_layout()
plt.show()

In [ ]:
# Violin plot: 클러스터별 각 score 분포
cluster_order = sorted(adata.obs['monocluster'].unique())
palette = sns.color_palette('tab10', n_colors=len(cluster_order))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, score, title in zip(
    axes,
    ['glycolysis_score', 'mito_energy_score'],
    ['Glycolysis Score', 'Mito Energy (OXPHOS) Score']
):
    sns.violinplot(
        data=adata.obs, x='monocluster', y=score,
        order=cluster_order, palette=palette,
        inner='box', scale='width', ax=ax
    )
    ax.set_title(title, fontsize=13)
    ax.set_xlabel('monocluster')
    ax.set_ylabel('Score')
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
from scipy import stats
from itertools import combinations
from statsmodels.stats.multitest import multipletests

def cluster_stats(obs_df, score_col):
    """Kruskal-Wallis + pairwise Mann-Whitney U (Bonferroni correction)"""
    groups = {cl: obs_df.loc[obs_df['monocluster'] == cl, score_col].values
              for cl in sorted(obs_df['monocluster'].unique())}

    # Kruskal-Wallis overall test
    kw_stat, kw_p = stats.kruskal(*groups.values())
    print(f'[{score_col}] Kruskal-Wallis: H={kw_stat:.4f}, p={kw_p:.2e}')

    # Pairwise Mann-Whitney U
    pairs = list(combinations(groups.keys(), 2))
    raw_p = []
    for a, b in pairs:
        _, p = stats.mannwhitneyu(groups[a], groups[b], alternative='two-sided')
        raw_p.append(p)

    _, adj_p, _, _ = multipletests(raw_p, method='bonferroni')

    results = pd.DataFrame({
        'cluster_A': [p[0] for p in pairs],
        'cluster_B': [p[1] for p in pairs],
        'p_raw': raw_p,
        'p_bonferroni': adj_p,
        'significant': adj_p < 0.05
    }).sort_values('p_bonferroni')

    return results

print('=' * 60)
glyco_stat = cluster_stats(adata.obs, 'glycolysis_score')
print(glyco_stat.to_string(index=False))

print('\n' + '=' * 60)
mito_stat = cluster_stats(adata.obs, 'mito_energy_score')
print(mito_stat.to_string(index=False))

In [ ]:
# Heatmap: 클러스터별 평균 score 요약
score_mean = adata.obs.groupby('monocluster')[['glycolysis_score', 'mito_energy_score']].mean()
score_mean.columns = ['Glycolysis', 'Mito Energy (OXPHOS)']
score_mean = score_mean.loc[sorted(score_mean.index)]

# Z-score normalization across clusters for better visual contrast
score_zscore = (score_mean - score_mean.mean()) / score_mean.std()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.heatmap(
    score_mean, annot=True, fmt='.3f', cmap='RdYlBu_r',
    linewidths=0.5, ax=axes[0], cbar_kws={'label': 'Mean Score'}
)
axes[0].set_title('Mean Activity Score per monocluster', fontsize=12)
axes[0].set_ylabel('monocluster')

sns.heatmap(
    score_zscore, annot=True, fmt='.2f', cmap='RdYlBu_r',
    linewidths=0.5, ax=axes[1], cbar_kws={'label': 'Z-score'}
)
axes[1].set_title('Z-score Normalized Score per monocluster', fontsize=12)
axes[1].set_ylabel('monocluster')

plt.tight_layout()
plt.show()

print('\n--- Mean scores per monocluster ---')
print(score_mean.round(4))

In [ ]:
# Warburg ratio: Glycolysis / Mito Energy (대사 전략 지표)
# 값이 클수록 해당 클러스터가 산화인산화보다 해당계를 선호함
eps = 1e-6
adata.obs['warburg_ratio'] = (
    adata.obs['glycolysis_score'] /
    (adata.obs['mito_energy_score'].abs() + eps)
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cluster_order = sorted(adata.obs['monocluster'].unique())
palette = sns.color_palette('tab10', n_colors=len(cluster_order))

# Warburg ratio violin
sns.violinplot(
    data=adata.obs, x='monocluster', y='warburg_ratio',
    order=cluster_order, palette=palette,
    inner='box', scale='width', ax=axes[0]
)
axes[0].set_title('Warburg Ratio (Glycolysis / Mito Energy) per monocluster', fontsize=11)
axes[0].set_xlabel('monocluster')
axes[0].set_ylabel('Warburg Ratio')
axes[0].tick_params(axis='x', rotation=45)
axes[0].axhline(y=adata.obs['warburg_ratio'].median(), color='red', linestyle='--', alpha=0.6, label='Median')
axes[0].legend()

# UMAP Warburg ratio
sc.pl.umap(adata, color='warburg_ratio', ax=axes[1], show=False,
           frameon=False, title='Warburg Ratio (UMAP)', color_map='RdYlBu_r')

plt.tight_layout()
plt.show()

print('\n--- Mean Warburg Ratio per monocluster ---')
print(adata.obs.groupby('monocluster')['warburg_ratio'].mean().sort_values(ascending=False).round(4))

In [ ]:
adata

## mono0 vs mono5: Fisher's Exact Test (Glycolysis & Mito Energy)
---

In [ ]:
from scipy.stats import fisher_exact

# mono0, mono5만 추출
df_sub = adata.obs[adata.obs['monocluster'].isin(['mono0', 'mono5'])][['monocluster', 'glycolysis_score', 'mito_energy_score']].copy()

# 전체 두 클러스터 통합 기준 median으로 High/Low 이진화
for score in ['glycolysis_score', 'mito_energy_score']:
    median_val = df_sub[score].median()
    df_sub[f'{score}_binary'] = df_sub[score].apply(lambda x: 'High' if x >= median_val else 'Low')
    print(f'{score} median threshold: {median_val:.4f}')

print()

results = []

for score, label in [('glycolysis_score', 'Glycolysis'), ('mito_energy_score', 'Mito Energy (OXPHOS)')]:
    col = f'{score}_binary'

    # 2x2 contingency table: rows = mono0/mono5, cols = High/Low
    ct = pd.crosstab(df_sub['monocluster'], df_sub[col])
    ct = ct.reindex(index=['mono0', 'mono5'], columns=['High', 'Low'], fill_value=0)

    odds_ratio, p_value = fisher_exact(ct.values)

    results.append({
        'Pathway': label,
        'mono0_High': ct.loc['mono0', 'High'],
        'mono0_Low':  ct.loc['mono0', 'Low'],
        'mono5_High': ct.loc['mono5', 'High'],
        'mono5_Low':  ct.loc['mono5', 'Low'],
        'Odds_Ratio': round(odds_ratio, 4),
        'p_value':    p_value,
        'Significant': p_value < 0.05
    })

    print(f'[{label}] Contingency Table:')
    print(ct)
    print(f'  Odds Ratio : {odds_ratio:.4f}')
    print(f'  p-value    : {p_value:.4e}  {"***" if p_value < 0.001 else ("**" if p_value < 0.01 else ("*" if p_value < 0.05 else "ns"))}')
    print()

result_df = pd.DataFrame(results)
print('=== Summary ===')
print(result_df.to_string(index=False))

In [ ]:
# mono0 vs mono5 score 분포 비교 시각화
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

palette_2 = {'mono0': '#2196F3', 'mono5': '#F44336'}

for ax, score, title in zip(
    axes,
    ['glycolysis_score', 'mito_energy_score'],
    ['Glycolysis Score', 'Mito Energy (OXPHOS) Score']
):
    sns.violinplot(
        data=df_sub, x='monocluster', y=score,
        order=['mono0', 'mono5'], palette=palette_2,
        inner='box', scale='width', ax=ax
    )
    # p-value annotation
    row = result_df[result_df['Pathway'] == title.replace(' Score', '')]
    if not row.empty:
        p = row['p_value'].values[0]
        sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
        y_max = df_sub[score].quantile(0.95)
        ax.plot([0, 1], [y_max, y_max], color='black', linewidth=1)
        ax.text(0.5, y_max * 1.02, f'{sig}\np={p:.2e}', ha='center', va='bottom', fontsize=10)
    ax.set_title(f'{title}\nmono0 vs mono5', fontsize=12)
    ax.set_xlabel('')

plt.tight_layout()
plt.show()

## SLC Family
-------------------

In [ ]:
genes = ['SLC7A5','LAT2','SLC43A1','SLC43A2','SLC3A2','SLC1A5']  # 예시 유전자 리스트
n_genes = len(genes)

fig, axs = plt.subplots(n_genes, 1, figsize=(6, 1.8 * n_genes))  # 세로 크기를 줄임

# axs가 하나일 경우 (genes가 1개) 대비
if n_genes == 1:
    axs = [axs]

for i, gene in enumerate(genes):
    sc.pl.violin(
        adata,
        keys=gene,
        groupby='monocluster',
        ax=axs[i],
        show=False,
        stripplot=False,
        jitter=False
    )
    axs[i].set_title(gene)

plt.tight_layout()
#plt.savefig("/home/jaehyunchoi/bystander_collaboration/IPF_data/analysis/figures/CD4+9cluster_violinplot.pdf", bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
genes = ['FCGR1A','SIGLEC1','CD68','MRC1','CCR2','S100A8','S100A9','CX3CR1']  # 예시 유전자 리스트
n_genes = len(genes)

fig, axs = plt.subplots(n_genes, 1, figsize=(6, 1.8 * n_genes))  # 세로 크기를 줄임

# axs가 하나일 경우 (genes가 1개) 대비
if n_genes == 1:
    axs = [axs]

for i, gene in enumerate(genes):
    sc.pl.violin(
        adata,
        keys=gene,
        groupby='monocluster',
        ax=axs[i],
        show=False,
        stripplot=False,
        jitter=False
    )
    axs[i].set_title(gene)

plt.tight_layout()
#plt.savefig("/home/jaehyunchoi/bystander_collaboration/IPF_data/analysis/figures/CD4+9cluster_violinplot.pdf", bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
genes = ['RPS6', 'EIF4EBP1']  # 예시 유전자 리스트
n_genes = len(genes)

fig, axs = plt.subplots(n_genes, 1, figsize=(6, 1.8 * n_genes))  # 세로 크기를 줄임

# axs가 하나일 경우 (genes가 1개) 대비
if n_genes == 1:
    axs = [axs]

for i, gene in enumerate(genes):
    sc.pl.violin(
        adata,
        keys=gene,
        groupby='monocluster',
        ax=axs[i],
        show=False,
        stripplot=False,
        jitter=False
    )
    axs[i].set_title(gene)

plt.tight_layout()
#plt.savefig("/home/jaehyunchoi/bystander_collaboration/IPF_data/analysis/figures/CD4+9cluster_violinplot.pdf", bbox_inches='tight', dpi=300)
plt.show()